# DMAPRIME DLM — Model Evaluation
FluSight DLM (UnobservedComponents) recursive evaluation, h=0..3, hospital admissions (SC + US), with Prisma + MUSC EHR positive-test and inpatient lags.

Run cells top to bottom.

In [ ]:
# =============================================================================
# FluSight — DLM (UnobservedComponents) RECURSIVE EVALUATION (h=0..3) — HOSP ONLY (SC + US)
#
#   Use EHR Positive Tests lagged values as features for BOTH SC and US models
#   ALSO use EHR Weekly_Inpatient_Hospitalizations lagged values as features
#   Use the SAME Prisma + MUSC directories 
#   Optimize y-lags, pos-test lags, and inpatient-hosp lags (1..10 templates) using VALID MAE across all horizons
#
# IMPORTANT NOTE ABOUT US:
#   The Prisma/MUSC EHR feeds you load here are SC-only. To still use EHR covariates for US,
#   this script applies the SAME combined SC EHR series (pos + inpatient) to BOTH SC and US
#   when APPLY_SC_EHR_TO_US=True.
#
# Requirements:
#   pip install pandas numpy matplotlib statsmodels
# =============================================================================

import os
import glob
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.structural import UnobservedComponents

warnings.filterwarnings("ignore")

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
base_dir = r"C:\Users\mdsakhh\Box\BoxPHI-PHMR Projects\Sakhawat\FluSight_Forecast"

FILE_HOSP = (
    "https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/"
    "main/target-data/target-hospital-admissions.csv"
)

# --- EHR Data paths (from your ARIMAX script) ---
PRISMA_DIR = r"C:\Users\mdsakhh\Box\BoxPHI-PHMR Projects\Data\Prisma Health\Infectious Disease EHR\Weekly Data\Latest Weekly Data"
PRISMA_BASENAME = "Prisma_Health_Weekly_Influenza_State_dx_cond_lab_Incident"
MUSC_DIR = r"C:\Users\mdsakhh\Box\BoxPHI-PHMR Projects\Data\MUSC\Infectious Disease EHR\Weekly Data\Latest Weekly Data"
MUSC_BASENAME = "MUSC_Weekly_Influenza_State_dx_cond_lab_Incident"

# Apply SC EHR series to US as well (since EHR feeds here are SC-only)
APPLY_SC_EHR_TO_US = False

LOCATIONS = [
    dict(code="45", name="South Carolina"),
    dict(code="US", name="US"),
]

OUTCOME_MEASURE = "wk inc flu hosp"
TARGET_KEY = "hosp"
HORIZONS = [0, 1, 2, 3]

USE_LOG_TRANSFORM = True  # log1p/expm1 on y AND EHR covariates (pos + inpatient)

REFERENCE_DATE_LABEL = pd.to_datetime("2026-02-21")  # eval-file label

# DLM candidate structures
DLM_STRUCTURES = [
    dict(name="level_only", level=True, trend=False, seasonal=None),
    dict(name="level_trend", level=True, trend=True, seasonal=None),
    dict(name="level_only_no_seas", level=True, trend=False, seasonal=None),
    dict(name="level_trend_no_seas", level=True, trend=True, seasonal=None),
]

# Optimize lags from 1..10 using templates 
Y_LAG_TEMPLATES = [
    [1, 2, 3,4],
    [1, 2, 3, 4, 5],
    [1, 2, 3, 4, 5, 6, 7],
    [1, 2, 4, 6, 8, 10],
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
]

POS_LAG_TEMPLATES = [
    [4, 5],
    [4, 5, 6],
    [4, 5, 6, 7, 8, 9],
    [4, 6, 8, 10],
    [4, 5, 6, 7, 8, 9, 10],
]

INP_LAG_TEMPLATES = [
    [4, 5],
    [4, 5, 6],
    [4, 5, 6, 7, 8, 9],
    [4, 6, 8, 10],
    [4, 5, 6, 7, 8, 9, 10],
]

# Splits
TRAIN_END = pd.to_datetime("2024-06-29")
VALID_END = pd.to_datetime("2025-01-31")
TEST_END  = pd.to_datetime("2025-04-26")

MIN_TRAIN_ROWS = 60

# Outputs
out_eval_dir   = os.path.join(base_dir, "model_eval", "eval_files")
out_plot_dir   = os.path.join(base_dir, "model_eval", "plots", "dlm_recursive_hosp_with_pos_inp_prisma_musc")
out_params_dir = os.path.join(base_dir, "model_eval", "optimal_parameters")
os.makedirs(out_eval_dir, exist_ok=True)
os.makedirs(out_plot_dir, exist_ok=True)
os.makedirs(out_params_dir, exist_ok=True)

In [ ]:
# =============================================================================
# UTILS
# =============================================================================
def _latest_file_by_mtime(file_list):
    if not file_list:
        return None
    return max(file_list, key=lambda p: os.path.getmtime(p))

def pick_col(df, candidates):
    nm = set(df.columns)
    for c in candidates:
        if c in nm:
            return c
    return None

def fmt_mdY(dt_like):
    dt = pd.to_datetime(dt_like)
    if isinstance(dt, pd.Series):
        s = dt.dt.strftime("%m/%d/%Y")
        s = s.str.replace(r"^0", "", regex=True)
        s = s.str.replace(r"/0", "/", regex=True)
        return s
    s = dt.strftime("%m/%d/%Y")
    mm, dd, yy = s.split("/")
    return f"{int(mm)}/{int(dd)}/{yy}"

def log1p_safe(x):
    return np.log1p(np.maximum(0.0, x))

def inv_log1p(x):
    return np.expm1(x)

def to_scalar(x):
    return float(np.asarray(x).ravel()[0])

def clip0(x):
    return float(max(0.0, x))

In [ ]:
# =============================================================================
# HOLIDAY FEATURES (target-date based)
# =============================================================================
def thanksgiving_flag(d):
    d = pd.Timestamp(d)
    if d.month != 11:
        return 0
    first = pd.Timestamp(year=d.year, month=11, day=1)
    offset = (3 - first.dayofweek) % 7  # Thu=3
    first_thu = first + pd.Timedelta(days=offset)
    fourth_thu = first_thu + pd.Timedelta(days=21)
    return int(abs((d - fourth_thu).days) <= 3)

def christmas_flag(d):
    d = pd.Timestamp(d)
    return int(d.month == 12 and d.day >= 20)

def newyear_flag(d):
    d = pd.Timestamp(d)
    return int(d.month == 1 and d.day <= 7)

In [ ]:
# =============================================================================
# LOAD TARGET SERIES
# =============================================================================
def load_target_series_hosp(url, location_code, outcome_measure):
    df = pd.read_csv(url)

    if "target_end_date" in df.columns:
        df["date"] = pd.to_datetime(df["target_end_date"])
    elif "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
    else:
        raise ValueError("Could not find date column in target data")

    df["location"] = df["location"].astype(str)
    sub = df[df["location"] == str(location_code)].copy()

    if "outcome_measure" in sub.columns:
        sub = sub[sub["outcome_measure"] == outcome_measure].copy()

    sub["y_original"] = pd.to_numeric(sub["value"], errors="coerce")
    sub = (
        sub[["date", "y_original"]]
        .dropna()
        .sort_values("date")
        .drop_duplicates("date")
        .reset_index(drop=True)
    )

    sub["y"] = sub["y_original"].astype(float)
    if USE_LOG_TRANSFORM:
        sub["y"] = log1p_safe(sub["y"].values.astype(float))

    return sub

# =============================================================================
# LOAD EHR COVARIATES (Prisma + MUSC combined) — SC
#   Returns: df with columns ["date","pos","inp"] (optionally log-space)
# =============================================================================
def load_ehr_covariates_sc():
    prisma_files = glob.glob(os.path.join(PRISMA_DIR, f"{PRISMA_BASENAME}*.csv"))
    musc_files   = glob.glob(os.path.join(MUSC_DIR,   f"{MUSC_BASENAME}*.csv"))

    def _load_one(path, tag):
        df = pd.read_csv(path)

        # Keep SC rows when available
        if "State" in df.columns:
            df = df[df["State"] == "SC"].copy()

        wk_col = pick_col(df, ["Week", "week", "Week_End", "week_end", "WeekEnd", "date", "Date"])
        if wk_col is None:
            raise ValueError(f"{tag}: week/date column not found")
        df[wk_col] = pd.to_datetime(df[wk_col])

        pos_col = pick_col(df, ["Weekly_Positive_Tests", "Weekly_Positive", "Positive_Tests", "Weekly_Positive_Tests_All"])
        inp_col = pick_col(df, ["Weekly_Inpatient_Hospitalizations", "Weekly_Inpatient_Hosp", "Inpatient_Hospitalizations"])

        if pos_col is None:
            df["__ehr_pos__"] = 0.0
            pos_col = "__ehr_pos__"
        if inp_col is None:
            df["__ehr_inp__"] = 0.0
            inp_col = "__ehr_inp__"

        out = df[[wk_col, pos_col, inp_col]].copy()
        out.columns = ["date", f"{tag}_pos", f"{tag}_inp"]
        out[f"{tag}_pos"] = pd.to_numeric(out[f"{tag}_pos"], errors="coerce").fillna(0.0)
        out[f"{tag}_inp"] = pd.to_numeric(out[f"{tag}_inp"], errors="coerce").fillna(0.0)
        return out

    pf = _latest_file_by_mtime(prisma_files)
    mf = _latest_file_by_mtime(musc_files)

    prisma_df, musc_df = None, None
    if pf is not None:
        print(f"Loading Prisma EHR: {os.path.basename(pf)}")
        prisma_df = _load_one(pf, "prisma")
    if mf is not None:
        print(f"Loading MUSC EHR:   {os.path.basename(mf)}")
        musc_df = _load_one(mf, "musc")

    if prisma_df is None and musc_df is None:
        print("WARNING: No EHR files found (Prisma/MUSC). Using zeros for EHR covariates.")
        return None

    if prisma_df is not None and musc_df is not None:
        ehr = pd.merge(prisma_df, musc_df, on="date", how="outer")
    elif prisma_df is not None:
        ehr = prisma_df.copy()
        ehr["musc_pos"] = 0.0
        ehr["musc_inp"] = 0.0
    else:
        ehr = musc_df.copy()
        ehr["prisma_pos"] = 0.0
        ehr["prisma_inp"] = 0.0

    ehr = ehr.fillna(0.0).sort_values("date").reset_index(drop=True)
    ehr["pos"] = ehr["prisma_pos"] + ehr["musc_pos"]
    ehr["inp"] = ehr["prisma_inp"] + ehr["musc_inp"]

    if USE_LOG_TRANSFORM:
        ehr["pos"] = log1p_safe(ehr["pos"].values.astype(float))
        ehr["inp"] = log1p_safe(ehr["inp"].values.astype(float))

    print(f"Combined EHR weeks: {len(ehr)} | {ehr['date'].min().date()} to {ehr['date'].max().date()}")
    return ehr[["date", "pos", "inp"]].copy()

In [ ]:
# =============================================================================
# BUILD 1-STEP DATASET WITH EHR POS + INPATIENT: origin t -> target t+1
# =============================================================================
def build_one_step_dataset_with_ehr(series_df, ehr_df, y_lags, pos_lags, inp_lags):
    df = series_df.copy().sort_values("date").reset_index(drop=True)

    if ehr_df is not None:
        df = pd.merge(df, ehr_df, on="date", how="left")
        df["pos"] = df["pos"].fillna(0.0)
        df["inp"] = df["inp"].fillna(0.0)
    else:
        df["pos"] = 0.0
        df["inp"] = 0.0

    rows = []
    for i in range(len(df) - 1):
        origin_date = df.loc[i, "date"]
        target_date = df.loc[i + 1, "date"]

        y_t = float(df.loc[i + 1, "y"])          # transformed
        y_o = float(df.loc[i + 1, "y_original"]) # original

        feats = {}

        for L in y_lags:
            j = i - L + 1
            feats[f"y_lag{L}"] = float(df.loc[j, "y"]) if j >= 0 else float(df.loc[0, "y"])

        for L in pos_lags:
            j = i - L + 1
            feats[f"pos_lag{L}"] = float(df.loc[j, "pos"]) if j >= 0 else float(df.loc[0, "pos"])

        for L in inp_lags:
            j = i - L + 1
            feats[f"inp_lag{L}"] = float(df.loc[j, "inp"]) if j >= 0 else float(df.loc[0, "inp"])

        feats["is_thanksgiving"] = thanksgiving_flag(target_date)
        feats["is_christmas"]    = christmas_flag(target_date)
        feats["is_newyear"]      = newyear_flag(target_date)

        row = {"origin_date": origin_date, "target_end_date": target_date, "y": y_t, "y_original": y_o}
        row.update(feats)
        rows.append(row)

    ds = pd.DataFrame(rows)

    exog_cols = [c for c in ds.columns if c.startswith(("y_lag", "pos_lag", "inp_lag", "is_"))]
    exog_cols = sorted(
        exog_cols,
        key=lambda x: (
            0 if x.startswith("y_lag") else
            1 if x.startswith("pos_lag") else
            2 if x.startswith("inp_lag") else
            3,
            x
        ),
    )
    return ds, exog_cols, df

In [ ]:
# =============================================================================
# FIT DLM
# =============================================================================
def fit_dlm(endog, exog, structure):
    model = UnobservedComponents(
        endog=endog,
        exog=exog,
        level=structure["level"],
        trend=structure["trend"],
        seasonal=structure["seasonal"],
    )
    res = model.fit(disp=False)
    return res

# =============================================================================
# RECURSIVE EVALUATION (h=0..3) USING 1-STEP MODEL WITH EHR (pos + inp)
#   - y is updated recursively
#   - EHR covariates are NOT updated (future unknown) => hold last observed values
# =============================================================================
def evaluate_recursive_with_ehr(series_df, ehr_df, cfg):
    y_lags = cfg["y_lags"]
    pos_lags = cfg["pos_lags"]
    inp_lags = cfg["inp_lags"]
    structure = cfg["structure"]

    ds1, exog_cols, df_full = build_one_step_dataset_with_ehr(series_df, ehr_df, y_lags, pos_lags, inp_lags)

    train_mask = ds1["target_end_date"] <= TRAIN_END
    valid_mask = (ds1["target_end_date"] > TRAIN_END) & (ds1["target_end_date"] <= VALID_END)
    test_mask  = (ds1["target_end_date"] > VALID_END) & (ds1["target_end_date"] <= TEST_END)

    if train_mask.sum() < MIN_TRAIN_ROWS or valid_mask.sum() < 10:
        return None

    # Fit TRAIN (train+valid recursion)
    ds_tr = ds1.loc[train_mask].copy()
    y_tr = ds_tr["y"].values.astype(float)
    X_tr = ds_tr[exog_cols].values.astype(float)
    res_tr = fit_dlm(y_tr, X_tr, structure)

    # Fit TRAIN+VALID (test recursion)
    tv_mask = train_mask | valid_mask
    ds_tv = ds1.loc[tv_mask].copy()
    y_tv = ds_tv["y"].values.astype(float)
    X_tv = ds_tv[exog_cols].values.astype(float)
    res_tv = fit_dlm(y_tv, X_tv, structure)

    df = df_full.copy()
    y_full   = df["y"].values.astype(float)
    pos_full = df["pos"].values.astype(float)
    inp_full = df["inp"].values.astype(float)

    dates_full = pd.to_datetime(df["date"])
    date_to_idx = {pd.Timestamp(d): i for i, d in enumerate(dates_full)}

    origins_train = pd.to_datetime(ds1.loc[train_mask, "origin_date"]).tolist()
    origins_valid = pd.to_datetime(ds1.loc[valid_mask, "origin_date"]).tolist()
    origins_test  = pd.to_datetime(ds1.loc[test_mask,  "origin_date"]).tolist()

    def lag_value(hist, idx, L):
        j = idx - L + 1
        return float(hist[j]) if j >= 0 else float(hist[0])

    def run_split(res_fit, origin_dates, split_name):
        rows = []
        for origin_date in origin_dates:
            origin_date = pd.Timestamp(origin_date)
            if origin_date not in date_to_idx:
                continue
            origin_idx = date_to_idx[origin_date]

            y_hist   = y_full[: origin_idx + 1].copy()
            pos_hist = pos_full[: origin_idx + 1].copy()
            inp_hist = inp_full[: origin_idx + 1].copy()

            for h in HORIZONS:
                step_k = h + 1
                target_date = origin_date + pd.Timedelta(days=7 * step_k)
                cur_idx = origin_idx + (step_k - 1)

                ex = {}
                for L in y_lags:
                    ex[f"y_lag{L}"] = lag_value(y_hist, cur_idx, L)
                for L in pos_lags:
                    ex[f"pos_lag{L}"] = lag_value(pos_hist, cur_idx, L)
                for L in inp_lags:
                    ex[f"inp_lag{L}"] = lag_value(inp_hist, cur_idx, L)

                ex["is_thanksgiving"] = thanksgiving_flag(target_date)
                ex["is_christmas"]    = christmas_flag(target_date)
                ex["is_newyear"]      = newyear_flag(target_date)

                x_row = np.array([[ex.get(c, 0.0) for c in exog_cols]], dtype=float)
                mu = to_scalar(res_fit.get_forecast(steps=1, exog=x_row).predicted_mean)

                # recursive update (y only)
                y_hist = np.append(y_hist, mu)

                # hold EHR covariates at last observed value (future unknown)
                pos_hist = np.append(pos_hist, float(pos_hist[-1]) if len(pos_hist) else 0.0)
                inp_hist = np.append(inp_hist, float(inp_hist[-1]) if len(inp_hist) else 0.0)

                if pd.Timestamp(target_date) not in date_to_idx:
                    continue
                ti = date_to_idx[pd.Timestamp(target_date)]
                y_obs_orig = float(df.loc[ti, "y_original"])

                y_pred_orig = inv_log1p(mu) if USE_LOG_TRANSFORM else mu
                y_pred_orig = clip0(y_pred_orig)

                rows.append({
                    "origin_date": origin_date,
                    "target_end_date": target_date,
                    "horizon": int(h),
                    "split": split_name,
                    "y_obs": y_obs_orig,
                    "y_pred": y_pred_orig,
                })
        return pd.DataFrame(rows)

    out_train = run_split(res_tr, origins_train, "train")
    out_valid = run_split(res_tr, origins_valid, "valid")
    out_test  = run_split(res_tv, origins_test,  "test")

    out = pd.concat([out_train, out_valid, out_test], ignore_index=True)
    return out if len(out) > 0 else None

In [ ]:
# =============================================================================
# SELECT BEST CONFIG (VALID MAE across all horizons)
# =============================================================================
def select_best(series, ehr_df):
    best = None
    best_out = None

    for structure in DLM_STRUCTURES:
        for y_lags in Y_LAG_TEMPLATES:
            for pos_lags in POS_LAG_TEMPLATES:
                for inp_lags in INP_LAG_TEMPLATES:
                    cfg = {
                        "y_lags": list(y_lags),
                        "pos_lags": list(pos_lags),
                        "inp_lags": list(inp_lags),
                        "structure": structure,
                    }
                    try:
                        out = evaluate_recursive_with_ehr(series, ehr_df, cfg)
                        if out is None or len(out) == 0:
                            continue
                        v = out[out["split"] == "valid"]
                        if len(v) == 0:
                            continue
                        mae = float(np.mean(np.abs(v["y_obs"] - v["y_pred"])))
                        rmse = float(np.sqrt(np.mean((v["y_obs"] - v["y_pred"]) ** 2)))
                        cand = {"valid_mae": mae, "valid_rmse": rmse, **cfg}
                        if (best is None) or (mae < best["valid_mae"]):
                            best = cand
                            best_out = out
                    except Exception:
                        continue

    return best, best_out

In [ ]:
# =============================================================================
# PLOT
# =============================================================================
def save_panel_plot(loc_name, results_df, fig_path):
    fig, axes = plt.subplots(2, 2, figsize=(18, 11))
    axes = axes.flatten()

    for idx, h in enumerate(HORIZONS):
        ax = axes[idx]
        dh = results_df[results_df["horizon"] == h].sort_values("target_end_date")
        if len(dh) == 0:
            ax.set_title(f"h={h} (no data)")
            continue

        ax.plot(dh["target_end_date"], dh["y_obs"], linewidth=2.4, marker="o", markersize=2, label="Observed")

        for split in ["train", "valid", "test"]:
            m = dh["split"] == split
            if m.sum() > 0:
                ax.plot(
                    dh.loc[m, "target_end_date"], dh.loc[m, "y_pred"],
                    linewidth=2.0, alpha=0.85, marker="s", markersize=2, label=split.capitalize()
                )

        ax.axvline(TRAIN_END, linestyle="--", linewidth=1.5, alpha=0.6)
        ax.axvline(VALID_END, linestyle="--", linewidth=1.5, alpha=0.6)

        txt = []
        for split in ["train", "valid", "test"]:
            m = dh["split"] == split
            if m.sum() > 0:
                mae = np.mean(np.abs(dh.loc[m, "y_obs"] - dh.loc[m, "y_pred"]))
                txt.append(f"{split}: MAE={mae:.1f}")
        ax.set_title(f"h={h} (REC {h+1}wk)\n" + " | ".join(txt), fontsize=10)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8, loc="best")

    plt.suptitle(
        f"{loc_name} | {OUTCOME_MEASURE} | DLM RECURSIVE (+POS + INP EHR LAGS) (h=0..3)",
        fontsize=15, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"✓ Saved plot: {fig_path}")

In [ ]:
# =============================================================================
# MAIN
# =============================================================================
print("\n" + "="*110)
print("DLM RECURSIVE EVALUATION — HOSP (SC + US) — ADD EHR POS + INP LAGS (Prisma+MUSC)")
print("="*110)
print(f"USE_LOG_TRANSFORM: {USE_LOG_TRANSFORM}")
print(f"APPLY_SC_EHR_TO_US: {APPLY_SC_EHR_TO_US}")
print(f"REFERENCE_DATE_LABEL (eval file): {REFERENCE_DATE_LABEL.date()}")
print(
    f"DLM structures: {len(DLM_STRUCTURES)} | y-lag templates: {len(Y_LAG_TEMPLATES)} | "
    f"pos-lag templates: {len(POS_LAG_TEMPLATES)} | inp-lag templates: {len(INP_LAG_TEMPLATES)}"
)
print("="*110)

ehr_sc = load_ehr_covariates_sc()

eval_rows = []
all_best_cfg = {}

for loc in LOCATIONS:
    loc_code = str(loc["code"])
    loc_name = loc["name"]
    loc_gen  = "State" if loc_code != "US" else "National"

    series = load_target_series_hosp(FILE_HOSP, loc_code, OUTCOME_MEASURE)
    print("\n" + "="*110)
    print(f"LOCATION: {loc_name} ({loc_code}) | weeks={len(series)} | {series['date'].min().date()} to {series['date'].max().date()}")
    print("="*110)

    # choose EHR series
    if loc_code == "45":
        ehr_use = ehr_sc
    else:
        ehr_use = ehr_sc if APPLY_SC_EHR_TO_US else None
        if (ehr_use is not None) and APPLY_SC_EHR_TO_US:
            print("NOTE: Using SC Prisma+MUSC EHR series (pos+inp) as exogenous input for US (per request).")

    best_cfg, best_out = select_best(series, ehr_use)

    if best_cfg is None or best_out is None:
        print("SKIP: no feasible config.")
        continue

    print("\n✓ Best config:")
    print(f"  structure={best_cfg['structure']}")
    print(f"  y_lags={best_cfg['y_lags']}")
    print(f"  pos_lags={best_cfg['pos_lags']}")
    print(f"  inp_lags={best_cfg['inp_lags']}")
    print(f"  VALID MAE={best_cfg['valid_mae']:.2f} | RMSE={best_cfg['valid_rmse']:.2f}")

    cfg_key = f"{loc_code}_{TARGET_KEY}_dlm_recursive_with_pos_inp_prisma_musc"
    all_best_cfg[cfg_key] = {
        "location": loc_code,
        "target_type": TARGET_KEY,
        "model_kind": "dlm_unobservedcomponents",
        "use_log_transform": bool(USE_LOG_TRANSFORM),
        "apply_sc_ehr_to_us": bool(APPLY_SC_EHR_TO_US),
        "y_lags": best_cfg["y_lags"],
        "pos_lags": best_cfg["pos_lags"],
        "inp_lags": best_cfg["inp_lags"],
        "structure": best_cfg["structure"],
        "valid_mae": float(best_cfg["valid_mae"]),
        "valid_rmse": float(best_cfg["valid_rmse"]),
        "timestamp": datetime.now().isoformat(),
    }

    fig_path = os.path.join(out_plot_dir, f"{loc_code}_{TARGET_KEY}_dlm_recursive_with_pos_inp_panel.png")
    save_panel_plot(loc_name, best_out, fig_path)

    # ---- eval rows (exact format) ----
    for _, r in best_out.iterrows():
        split = str(r["split"])
        eval_rows.append({
            "reference_date": REFERENCE_DATE_LABEL,
            "target": "NA",
            "location_general": loc_gen,
            "location": loc_name,
            "target_end_date": pd.to_datetime(r["target_end_date"]),
            "value": float(r["y_pred"]),
            "disease": "influenza",
            "population": "general",
            "training_validation": int(1 if split in ["train", "valid"] else 0),
            "estimate_projected_report": int(r["horizon"]),
            "imputed": 0,
            "data_source": "CDC NHSN",
            "outcome_measure": OUTCOME_MEASURE,
            "output_type": "quantile",
            "output_type_id": "0.5",
        })

In [ ]:
# =============================================================================
# SAVE EVAL CSV
# =============================================================================
if len(eval_rows) > 0:
    eval_df = pd.DataFrame(eval_rows)

    col_order = [
        "reference_date",
        "target",
        "location_general",
        "location",
        "target_end_date",
        "value",
        "disease",
        "population",
        "training_validation",
        "estimate_projected_report",
        "imputed",
        "data_source",
        "outcome_measure",
        "output_type",
        "output_type_id",
    ]
    eval_df = eval_df[col_order].sort_values(
        ["location", "outcome_measure", "estimate_projected_report", "target_end_date"]
    ).reset_index(drop=True)

    eval_df["reference_date"] = fmt_mdY(eval_df["reference_date"])
    eval_df["target_end_date"] = fmt_mdY(eval_df["target_end_date"])

    suffix = "log" if USE_LOG_TRANSFORM else "raw"
    out_eval_path = os.path.join(out_eval_dir, f"eval_dlm_recursive_hosp_with_pos_inp_prisma_musc_{suffix}_FORMAT.csv")
    eval_df.to_csv(out_eval_path, index=False)
    print(f"\n✓ Saved evaluation (format-ready): {out_eval_path}")

In [ ]:
# =============================================================================
# SAVE OPTIMAL PARAMS JSON
# =============================================================================
suffix = "log" if USE_LOG_TRANSFORM else "raw"
out_cfg_path = os.path.join(out_params_dir, f"optimal_params_dlm_recursive_hosp_with_pos_inp_prisma_musc_{suffix}.json")
with open(out_cfg_path, "w") as f:
    json.dump(all_best_cfg, f, indent=2)
print(f"✓ Saved best DLM configs: {out_cfg_path}")

print("\n" + "="*110)
print("DONE — DLM RECURSIVE EVALUATION (+POS + INPATIENT EHR LAGS) — Prisma+MUSC directories")
print("="*110)